# Odev2: CIFAR-10 Goruntu Siniflandirma

Bu notebook, PyTorch kullanilarak tasarlanmis CNN modellerini ve klasik makine ogrenmesi hibritlerini (Odev1 standartlarina uygun, tam OOP kapsullemesiyle) icermektedir.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '.')

from src.data_preprocessing import get_cifar10_loaders, CIFAR10_CLASSES
from src.cnn_models import BasicCNN, ImprovedCNN, get_vgg16_transfer
from src.classifier import CNNClassifier
from src.hybrid import HybridClassifier
from src.metrics import plot_training_curves, evaluate_and_plot_cm

# Hiperparametreler
EPOCHS = 10
EPOCHS_VGG = 5
BATCH_SIZE = 128
LR = 0.001
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
print("Moduller ve kutuphaneler yuklendi.")

## 1. Veri Seti Hazirligi

In [ ]:
train_loader, test_loader = get_cifar10_loaders(batch_size=BATCH_SIZE)
classes = CIFAR10_CLASSES
print(f"Siniflar: {classes}")

## 2. Model 1 (Temel CNN) Egitimi

In [ ]:
# Model objesi ve OOP Classifier saricisi
model1_arch = BasicCNN(num_classes=10)
clf1 = CNNClassifier(model=model1_arch, lr=LR)

print("Model 1 egitiliyor...")
history1 = clf1.fit(train_loader, val_loader=test_loader, epochs=EPOCHS)
plot_training_curves(history1, title="Model 1 (Temel CNN)")

# Degerlendirme
y_true1, y_pred1 = clf1.predict(test_loader)
evaluate_and_plot_cm(y_true1, y_pred1, classes, title="Model 1 - Confusion Matrix")

## 3. Model 2 (Iyilestirilmis CNN) Egitimi

Dropout ve BatchNorm iceren yapi.

In [ ]:
model2_arch = ImprovedCNN(num_classes=10)
clf2 = CNNClassifier(model=model2_arch, lr=LR)

print("Model 2 egitiliyor...")
history2 = clf2.fit(train_loader, val_loader=test_loader, epochs=EPOCHS)
plot_training_curves(history2, title="Model 2 (Improved CNN)")

# Degerlendirme
y_true2, y_pred2 = clf2.predict(test_loader)
evaluate_and_plot_cm(y_true2, y_pred2, classes, title="Model 2 - Confusion Matrix")

## 4. Model 3 (VGG-16 Transfer Learning)

In [ ]:
# VGG-16 icin ozel loader (224x224)
train_loader_vgg, test_loader_vgg = get_cifar10_loaders(batch_size=64, resize_224=True)

model3_arch = get_vgg16_transfer(num_classes=10)
clf3 = CNNClassifier(model=model3_arch, lr=LR)

print("Model 3 (VGG-16) egitiliyor...")
history3 = clf3.fit(train_loader_vgg, val_loader=test_loader_vgg, epochs=EPOCHS_VGG)
plot_training_curves(history3, title="Model 3 (VGG-16)")

y_true3, y_pred3 = clf3.predict(test_loader_vgg)
evaluate_and_plot_cm(y_true3, y_pred3, classes, title="Model 3 - Confusion Matrix")

## 5. Model 4 (Hibrit - SVM ve Random Forest)

In [ ]:
# SVM Hibrit
hybrid_svm = HybridClassifier(feature_extractor=clf2.model, ml_type='svm')
hybrid_svm.fit(train_loader)

y_true_svm, y_pred_svm = hybrid_svm.predict(test_loader)
evaluate_and_plot_cm(y_true_svm, y_pred_svm, classes, title="Model 4 (Hibrit SVM)")

# RF Hibrit
hybrid_rf = HybridClassifier(feature_extractor=clf2.model, ml_type='rf')
hybrid_rf.fit(train_loader)

y_true_rf, y_pred_rf = hybrid_rf.predict(test_loader)
evaluate_and_plot_cm(y_true_rf, y_pred_rf, classes, title="Model 4 (Hibrit RF)")